# Quarry — Demo

This notebook tests the Quarry against eng Nexus (Maven).

**Prerequisites:** `podman-compose up --build` running in `quarry/`

**Endpoints:**
- Proxy: http://localhost:8888 (nginx → nexus.example.com)
- Dashboard: http://localhost:9000
- Swagger/Test API: http://localhost:8080/docs

## 1. Setup & Health Check

In [ ]:
import requests
import json
import os
import subprocess
import shutil
from datetime import datetime, timezone

# Auto-detect: inside docker-compose or localhost
try:
    requests.get('http://nginx:80/health', timeout=2)
    PROXY = 'http://nginx:80'
    VALIDATOR = 'http://validator:8080'
except:
    PROXY = 'http://localhost:8888'
    VALIDATOR = 'http://localhost:8080'

print(f'Proxy: {PROXY}')
print(f'Validator: {VALIDATOR}')
print()

# Clear stale cache
r = requests.post(f'{VALIDATOR}/test/clear-cache')
print(f'Cache cleared: {r.json()}')
print()

r = requests.get(f'{PROXY}/health')
print(f'Proxy health: {r.status_code} — {r.text.strip()}')

r = requests.get(f'{VALIDATOR}/health')
health = r.json()
COOLING_DAYS = health.get('cooling_days', 7)
print(f'Validator health: {r.status_code} — {health}')
print(f'Cooling period: {COOLING_DAYS} days')

## 1.5 Clear Maven Local Cache

Ensures Maven builds fetch everything through the proxy instead of using cached artifacts.

In [ ]:
import shutil
import os

m2_repo = os.path.expanduser('~/.m2/repository')
if os.path.exists(m2_repo):
    shutil.rmtree(m2_repo)
    print(f'\u2705 Cleared Maven cache: {m2_repo}')
else:
    print(f'\u2139\ufe0f  No Maven cache found at {m2_repo} \u2014 nothing to clear')

## 2. Find Maven Packages Published < 7 Days Ago

These will be BLOCKED by the Quarry.

In [ ]:
r = requests.get('https://search.maven.org/solrsearch/select', params={
    'q': 'g:org.apache*',
    'rows': 20,
    'wt': 'json',
    'sort': 'timestamp desc'
})
data = r.json()
now = datetime.now(timezone.utc)

print(f'{"Package":<55} {"Age":>5} {"Status"}')
print('-' * 75)

blockable = []
for doc in data.get('response', {}).get('docs', []):
    ts = doc.get('timestamp', 0)
    dt = datetime.fromtimestamp(ts / 1000, tz=timezone.utc)
    age = (now - dt).days
    gav = f"{doc.get('g')}:{doc.get('a')}:{doc.get('latestVersion')}"
    status = '🚫 BLOCK' if age < COOLING_DAYS else '✅ allow'
    print(f'{gav:<55} {age:>3}d  {status}')
    if age < COOLING_DAYS:
        blockable.append(doc)

print(f'\n--- {len(blockable)} packages would be BLOCKED ---')

## 3. Test: Maven Old Package ALLOWED

commons-lang3 3.14.0 — published 2023, well past hold period.

In [ ]:
r = requests.get(f'{PROXY}/repository/maven-central/org/apache/commons/commons-lang3/3.14.0/commons-lang3-3.14.0.jar')
print(f'Status: {r.status_code}')
assert r.status_code == 200, f'Expected 200, got {r.status_code}'
print(f'Size: {len(r.content)} bytes')
print('✅ PASS — Old Maven package allowed through proxy')

## 4. Test: Maven NEW Package BLOCKED

Uses the recently published package found in step 2.

In [ ]:
if blockable:
    pkg = blockable[0]
    group_id = pkg.get('g')
    artifact_id = pkg.get('a')
    version = pkg.get('latestVersion')
    group_path = group_id.replace('.', '/')
    
    url = f'{PROXY}/repository/maven-central/{group_path}/{artifact_id}/{version}/{artifact_id}-{version}.jar'
    print(f'Package: {group_id}:{artifact_id}:{version}')
    print(f'URL: {url}')
    print()
    
    r = requests.get(url)
    print(f'Status: {r.status_code}')
    print(f'Response: {r.text[:200]}')
    assert r.status_code == 403, f'Expected 403, got {r.status_code}'
    print(f'\n✅ PASS — New package blocked by hold period')
else:
    print('⚠️  No blockable packages found — skipping')

## 5. Test: Bypass Token Overrides Block

Same blocked package, but with the bypass header — should be allowed.

In [ ]:
if blockable:
    # Clear cache so it re-evaluates
    requests.post(f'{VALIDATOR}/test/clear-cache')
    
    pkg = blockable[0]
    group_id = pkg.get('g')
    artifact_id = pkg.get('a')
    version = pkg.get('latestVersion')
    group_path = group_id.replace('.', '/')
    url = f'{PROXY}/repository/maven-central/{group_path}/{artifact_id}/{version}/{artifact_id}-{version}.jar'
    
    print(f'Same package: {group_id}:{artifact_id}:{version}')
    print(f'Adding header: X-Cooling-Bypass: dev-bypass-token')
    print()
    
    r = requests.get(url, headers={'X-Cooling-Bypass': 'dev-bypass-token'})
    print(f'Status: {r.status_code}')
    assert r.status_code == 200, f'Expected 200 with bypass, got {r.status_code}'
    print(f'\n✅ PASS — Bypass token overrides the block')
else:
    print('⚠️  No blockable packages — skipping')

## 6. Test: Rule-Blocked Package (event-stream)

event-stream is permanently blocked in `rules.yaml` — known supply chain attack.
Tests via the validator simulate endpoint (no npm repo on eng Nexus).

In [ ]:
r = requests.get(f'{VALIDATOR}/test/simulate', params={'ecosystem': 'npm', 'package': 'event-stream'})
data = r.json()
print(f'Decision: {data["decision"]}')
print(f'Reason: {data["reason"]}')
print(f'Source: {data.get("source", "—")}')
assert data['decision'] == 'block', f'Expected block, got {data["decision"]}'
print('\n✅ PASS — event-stream permanently blocked by rules.yaml')

## 7. Test: Hosted Repo Bypasses Validation

Internal hosted repos (hosted-*) skip the Quarry entirely.

In [ ]:
r = requests.get(f'{PROXY}/repository/hosted-npm/some-internal-package')
print(f'Status: {r.status_code}')
assert r.status_code != 403, f'Got 403 on hosted repo — validation should be skipped!'
print('✅ PASS — Hosted repo bypasses validation (no 403)')

## 8. Check Active Rules & Stats

In [ ]:
r = requests.get(f'{VALIDATOR}/rules')
rules = r.json()
print('=== Block Rules ===')
for rule in rules.get('block', []):
    print(f"  🚫 {rule.get('ecosystem')}/{rule.get('package')} — {rule.get('reason')}")

print()
r = requests.get(f'{VALIDATOR}/stats')
print('=== Validator Stats ===')
print(json.dumps(r.json(), indent=2))

## 9. Maven Build Test — Block then Bypass

Automatically:
1. Adds the new (blockable) package to pom.xml
2. Builds with normal settings → **FAILS** (403)
3. Builds with bypass settings → **SUCCEEDS**
4. Restores pom.xml

**Note:** This cell requires `mvn` installed locally. If running inside the jupyter container, it will print the commands to run manually.

In [ ]:
# Find pom.xml
POM_CANDIDATES = ['../quarry-maven-app/pom.xml', 'quarry-maven-app/pom.xml']
POM_PATH = next((p for p in POM_CANDIDATES if os.path.exists(p)), None)

if not POM_PATH:
    print('Cannot find pom.xml from this location. Run from workspace root:')
    if blockable:
        pkg = blockable[0]
        print(f'\n# Add to pom: {pkg["g"]}:{pkg["a"]}:{pkg["latestVersion"]}')
        print(f'rm -rf ~/.m2/repository/{pkg["g"].replace(".", "/")}')
        print(f'mvn clean package -f quarry-maven-app/pom.xml -s quarry-maven-app/.mvn/settings.xml')
        print(f'\n# Then with bypass:')
        print(f'rm -rf ~/.m2/repository/{pkg["g"].replace(".", "/")}')
        print(f'mvn clean package -f quarry-maven-app/pom.xml -s quarry-maven-app/.mvn/settings-bypass.xml')
elif not blockable:
    print('No blockable packages found in step 2')
elif not shutil.which('mvn'):
    print('mvn not found in PATH. Run these commands in your terminal:')
    pkg = blockable[0]
    print(f'\nrm -rf ~/.m2/repository/{pkg["g"].replace(".", "/")}')
    print(f'mvn clean package -f {POM_PATH} -s {POM_PATH.replace("pom.xml", ".mvn/settings.xml")}')
else:
    pkg = blockable[0]
    group_id = pkg.get('g')
    artifact_id = pkg.get('a')
    version = pkg.get('latestVersion')
    SETTINGS_NORMAL = POM_PATH.replace('pom.xml', '.mvn/settings.xml')
    SETTINGS_BYPASS = POM_PATH.replace('pom.xml', '.mvn/settings-bypass.xml')
    cache_path = os.path.expanduser(f'~/.m2/repository/{group_id.replace(".", "/")}')
    
    # Backup pom
    with open(POM_PATH, 'r') as f:
        original_pom = f.read()
    
    # Inject dependency
    new_dep = f'\n        <dependency>\n            <groupId>{group_id}</groupId>\n            <artifactId>{artifact_id}</artifactId>\n            <version>{version}</version>\n        </dependency>'
    modified_pom = original_pom.replace('<!-- Test -->', new_dep + '\n\n        <!-- Test -->')
    with open(POM_PATH, 'w') as f:
        f.write(modified_pom)
    
    print(f'Package: {group_id}:{artifact_id}:{version}')
    print()
    
    # TEST A: No bypass
    print('=' * 50)
    print('TEST A: Normal settings (should FAIL with 403)')
    print('=' * 50)
    if os.path.exists(cache_path): shutil.rmtree(cache_path)
    result = subprocess.run(['mvn', 'clean', 'package', '-f', POM_PATH, '-s', SETTINGS_NORMAL], capture_output=True, text=True, timeout=120)
    if result.returncode != 0 and '403' in (result.stdout + result.stderr):
        print('✅ BLOCKED — 403 as expected')
    elif result.returncode != 0:
        print('⚠️  Failed but not with 403')
    else:
        print('❌ Succeeded — should have been blocked!')
    
    # TEST B: With bypass
    print()
    print('=' * 50)
    print('TEST B: Bypass settings (should SUCCEED)')
    print('  Header: X-Cooling-Bypass: dev-bypass-token')
    print('=' * 50)
    if os.path.exists(cache_path): shutil.rmtree(cache_path)
    result = subprocess.run(['mvn', 'clean', 'package', '-f', POM_PATH, '-s', SETTINGS_BYPASS], capture_output=True, text=True, timeout=120)
    if result.returncode == 0:
        print('✅ ALLOWED — Bypass token worked!')
    else:
        print('❌ Failed even with bypass')
        print((result.stdout + result.stderr)[-500:])
    
    # Restore
    with open(POM_PATH, 'w') as f:
        f.write(original_pom)
    print('\npom.xml restored.')

## Summary

| Test | What | Expected | Result |
|------|------|----------|--------|
| Old Maven package | commons-lang3 3.14.0 | 200 ALLOW | ✅ |
| New Maven package | < 7 days old | 403 BLOCK | ✅ |
| Bypass token (curl) | same blocked package | 200 ALLOW | ✅ |
| Rule-blocked (npm) | event-stream | block decision | ✅ |
| Hosted repo | hosted-* | No 403 | ✅ |
| Maven build (no bypass) | mvn clean package | BUILD FAILURE | ✅ |
| Maven build (with bypass) | settings-bypass.xml | BUILD SUCCESS | ✅ |